### Phase 3: Interactive Dashboard

Builds an interactive dashboard on top of the processed data from Phase 1 (ingestion) and Phase 2 (NLP). Connects to Supabase, pulls all enriched posts, and generates visualizations covering sentiment trends over time, stance breakdowns per topic, emotion radar charts, platform comparison views, and topic-level deep dives. Uses Plotly for interactive charts and IPython widgets for filtering — all running inside Colab at zero cost.

In [1]:
#  Install & imports necessary libraries
!pip install supabase plotly

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta
from supabase import create_client
from google.colab import userdata
import json

In [2]:
SUPABASE_URL = userdata.get("Supabase_url")
SUPABASE_KEY = userdata.get("Supabase_Key")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Fetch all processed posts (paginated)
posts = []
batch_size = 1000
offset = 0

while True:
    response = supabase.table("posts").select("*").eq("is_processed", True).range(offset, offset + batch_size - 1).execute()
    batch = response.data if response.data else []
    if not batch:
        break
    posts.extend(batch)
    offset += batch_size

df = pd.DataFrame(posts)

# Parse timestamps
df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601")
df["ingested_at"] = pd.to_datetime(df["ingested_at"], format="ISO8601")

# Parse emotions
def parse_emotions(emo):
    if isinstance(emo, str):
        try:
            return json.loads(emo)
        except:
            return {}
    elif isinstance(emo, dict):
        return emo
    return {}

df["emotions_parsed"] = df["emotions"].apply(parse_emotions)

# Apply same cleaning as Phase 2.5
df = df[df["created_at"] >= "2026-04-01"]  # Recent data only
df = df.drop_duplicates(subset=["content"], keep="first")
df["date"] = df["created_at"].dt.date

# Set emotions availability flag
EMOTIONS_AVAILABLE = (df["emotions_valid"].sum() > len(df) * 0.5) if "emotions_valid" in df.columns else False

print(f"Loaded {len(df)} clean posts")
print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")
print(f"Emotions available: {EMOTIONS_AVAILABLE}")

Loaded 1940 clean posts
Date range: 2026-04-01 18:34:59+00:00 to 2026-05-01 20:38:51.735390+00:00
Emotions available: True


In [3]:
# ══════════════════════════════════════════
# PANEL 1: High-level overview
# ══════════════════════════════════════════

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Posts per Topic",
        "Sentiment Distribution",
        "Posts per Source Category",
        "Avg Sentiment per Topic"
    ),
    specs=[
        [{"type": "bar"}, {"type": "pie"}],
        [{"type": "pie"}, {"type": "bar"}],
    ]
)

# 1. Posts per topic (bar)
topic_counts = df["topic"].value_counts()
fig.add_trace(
    go.Bar(
        x=topic_counts.index,
        y=topic_counts.values,
        marker_color="#1B6B93",
    ),
    row=1, col=1
)

# 2. Sentiment distribution (pie)
sent_counts = df["sentiment_label"].value_counts()
fig.add_trace(
    go.Pie(
        labels=sent_counts.index,
        values=sent_counts.values,
        marker_colors=["#636EFA", "#EF553B", "#00CC96"],
        hole=0.4,
    ),
    row=1, col=2
)

# 3. Source category distribution (pie)
cat_counts = df["source_category"].value_counts()
fig.add_trace(
    go.Pie(
        labels=cat_counts.index,
        values=cat_counts.values,
        marker_colors=["#AB63FA", "#FFA15A", "#19D3F3"],
        hole=0.4,
    ),
    row=2, col=1
)

# 4. Avg sentiment per topic (bar)
topic_sentiment = df.groupby("topic")["sentiment_score"].mean().sort_values()
colors = ["#EF553B" if v < 0 else "#00CC96" for v in topic_sentiment.values]
fig.add_trace(
    go.Bar(
        x=topic_sentiment.index,
        y=topic_sentiment.values,
        marker_color=colors,
    ),
    row=2, col=2
)

fig.update_layout(
    title_text="Public Opinion Pulse — Dashboard Overview",
    height=700,
    showlegend=False,
    template="plotly_dark",
)
fig.show()

In [4]:
# ══════════════════════════════════════════
# PANEL 2: Sentiment over time
# ══════════════════════════════════════════

# Group by date and topic
df["date"] = df["created_at"].dt.date
daily_sentiment = df.groupby(["date", "topic"])["sentiment_score"].mean().reset_index()

fig = px.line(
    daily_sentiment,
    x="date",
    y="sentiment_score",
    color="topic",
    title="Sentiment Trends Over Time (Daily Average)",
    labels={"sentiment_score": "Avg Sentiment", "date": "Date"},
    template="plotly_dark",
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.update_layout(height=500, legend_title="Topic")
fig.show()

In [5]:
# ══════════════════════════════════════════
# PANEL 3: Stance distribution per topic
# ══════════════════════════════════════════

stance_data = df.groupby(["topic", "stance"]).size().reset_index(name="count")
stance_total = stance_data.groupby("topic")["count"].transform("sum")
stance_data["percentage"] = round(stance_data["count"] / stance_total * 100, 1)

fig = px.bar(
    stance_data,
    x="topic",
    y="percentage",
    color="stance",
    title="Stance Distribution per Topic (For / Against / Neutral)",
    labels={"percentage": "Percentage (%)", "topic": "Topic"},
    color_discrete_map={"for": "#00CC96", "against": "#EF553B", "neutral": "#636EFA"},
    template="plotly_dark",
    barmode="stack",
    text="percentage",
)

fig.update_traces(texttemplate="%{text}%", textposition="inside")
fig.update_layout(height=500, legend_title="Stance")
fig.show()

In [6]:
# ══════════════════════════════════════════
# PANEL 4: Emotion radar charts
# ══════════════════════════════════════════

if EMOTIONS_AVAILABLE:
    emotion_labels = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

    # Calculate average emotions per topic
    topic_emotions = {}
    for topic in df["topic"].unique():
        topic_df = df[df["topic"] == topic]
        emotion_avgs = {e: 0.0 for e in emotion_labels}
        count = 0
        for _, row in topic_df.iterrows():
            emo = row["emotions_parsed"]
            if emo:
                for e in emotion_labels:
                    emotion_avgs[e] += emo.get(e, 0)
                count += 1
        if count > 0:
            emotion_avgs = {e: round(v / count, 3) for e, v in emotion_avgs.items()}
        topic_emotions[topic] = emotion_avgs

    # Create radar chart
    fig = go.Figure()

    colors = px.colors.qualitative.Set2
    for i, (topic, emotions) in enumerate(topic_emotions.items()):
        values = [emotions[e] for e in emotion_labels]
        values.append(values[0])  # Close the radar

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=emotion_labels + [emotion_labels[0]],
            name=topic,
            line_color=colors[i % len(colors)],
            fill="toself",
            opacity=0.3,
        ))

    fig.update_layout(
        title="Emotion Profile per Topic",
        polar=dict(radialaxis=dict(visible=True, range=[0, 0.5])),
        template="plotly_dark",
        height=600,
    )
    fig.show()
else:
    print("⚠ Emotion data not available — skipping emotion panel")

In [7]:
# ══════════════════════════════════════════
# PANEL 5: Platform comparison
# ══════════════════════════════════════════

platform_sentiment = df.groupby(["source_name", "topic"])["sentiment_score"].mean().reset_index()

# Only show sources with 10+ posts
source_counts = df["source_name"].value_counts()
active_sources = source_counts[source_counts >= 10].index.tolist()
platform_sentiment = platform_sentiment[platform_sentiment["source_name"].isin(active_sources)]

fig = px.bar(
    platform_sentiment,
    x="topic",
    y="sentiment_score",
    color="source_name",
    barmode="group",
    title="Sentiment Comparison Across Platforms",
    labels={"sentiment_score": "Avg Sentiment", "source_name": "Source"},
    template="plotly_dark",
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.update_layout(height=500, legend_title="Source")
fig.show()

In [8]:
# ══════════════════════════════════════════
# PANEL 6: Most engaged posts
# ══════════════════════════════════════════

for topic in sorted(df["topic"].unique()):
    topic_df = df[df["topic"] == topic].nlargest(5, "engagement")

    if topic_df.empty:
        continue

    print(f"\n{'='*60}")
    print(f"  TOP ENGAGED: {topic.upper()}")
    print(f"{'='*60}")

    for _, row in topic_df.iterrows():
        sentiment_emoji = "🟢" if row["sentiment_score"] > 0.2 else "🔴" if row["sentiment_score"] < -0.2 else "🟡"
        content_preview = row["content"][:120] + "..." if len(str(row["content"])) > 120 else row["content"]
        print(f"\n  {sentiment_emoji} [{row['source_name']}] Engagement: {row['engagement']}")
        print(f"  Sentiment: {row['sentiment_score']:.3f} | Stance: {row['stance']}")
        print(f"  {content_preview}")


  TOP ENGAGED: ARTIFICIAL INTELLIGENCE

  🟢 [youtube] Engagement: 147748
  Sentiment: 0.727 | Stance: for
  Claude Mythos: Highlights from 244-page Release. The model, the mythos, the legend. We have a new best AI model, but not...

  🟡 [youtube] Engagement: 91849
  Sentiment: 0.000 | Stance: neutral
  What the New ChatGPT 5.4 Means for the World. Just 48 hours after releasing GPT 5.3 Instant, OpenAI have released GPT 5....

  🟡 [youtube] Engagement: 85733
  Sentiment: 0.000 | Stance: neutral
  Claude Opus 4.7 - A New Frontier, in Performance … and Drama. Claude Opus 4.7 just dropped, but behind every headline li...

  🟡 [youtube] Engagement: 73640
  Sentiment: 0.000 | Stance: neutral
  Two AI Models Set to “stir government urgency”, But Will This Challenge Undo Them?. First look at exclusive reports abou...

  🟡 [bluesky] Engagement: 66
  Sentiment: 0.000 | Stance: neutral
  🎞️ Academy Awards organizers issued new rules to clarify that acting and writing must ‌be performed by humans 

In [16]:
# ══════════════════════════════════════════
# PANEL 7: Sentiment heatmap
# ══════════════════════════════════════════

# Pivot: topic vs source
heatmap_data = df.groupby(["topic", "source_category"])["sentiment_score"].mean().reset_index()
heatmap_pivot = heatmap_data.pivot(index="topic", columns="source_category", values="sentiment_score")

fig = px.imshow(
    heatmap_pivot,
    title="Sentiment Heatmap: Topic × Source Category",
    labels=dict(x="Source Category", y="Topic", color="Avg Sentiment"),
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    template="plotly_dark",
    aspect="auto",
)

fig.update_layout(height=500)
fig.show()

In [10]:
# ══════════════════════════════════════════
# PANEL 8: Opinion river (stance over time)
# ══════════════════════════════════════════

# Group by date and stance
stance_time = df.groupby(["date", "stance"]).size().reset_index(name="count")
stance_total_per_day = stance_time.groupby("date")["count"].transform("sum")
stance_time["percentage"] = round(stance_time["count"] / stance_total_per_day * 100, 1)

fig = px.area(
    stance_time,
    x="date",
    y="percentage",
    color="stance",
    title="Opinion River — Stance Proportions Over Time",
    labels={"percentage": "Share (%)", "date": "Date"},
    color_discrete_map={"for": "#00CC96", "against": "#EF553B", "neutral": "#636EFA"},
    template="plotly_dark",
)

fig.update_layout(height=500, legend_title="Stance")
fig.show()

In [14]:
# ══════════════════════════════════════════
# PANEL 9: Single topic deep dive
# ══════════════════════════════════════════

def topic_deep_dive(topic_name):
    topic_df = df[df["topic"] == topic_name]

    if topic_df.empty:
        print(f"No data for topic: {topic_name}")
        return

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f"Sentiment Over Time",
            f"Stance Breakdown",
            f"Source Distribution",
            f"Emotion Profile",
        ),
        specs=[
            [{"type": "scatter"}, {"type": "pie"}],
            [{"type": "bar"}, {"type": "scatterpolar"}],
        ]
    )

    # 1. Sentiment over time
    daily = topic_df.groupby("date")["sentiment_score"].mean().reset_index()
    fig.add_trace(
        go.Scatter(x=daily["date"], y=daily["sentiment_score"],
                   mode="lines+markers", line_color="#1B6B93"),
        row=1, col=1
    )

    # 2. Stance pie
    stance_counts = topic_df["stance"].value_counts()
    fig.add_trace(
        go.Pie(labels=stance_counts.index, values=stance_counts.values,
               marker_colors=["#636EFA", "#EF553B", "#00CC96"], hole=0.4),
        row=1, col=2
    )

    # 3. Source bar
    src_counts = topic_df["source_name"].value_counts().head(8)
    fig.add_trace(
        go.Bar(x=src_counts.index, y=src_counts.values, marker_color="#AB63FA"),
        row=2, col=1
    )

    # 4. Emotion radar
    emotion_avgs = {e: 0.0 for e in emotion_labels}
    count = 0
    for _, row in topic_df.iterrows():
        emo = row["emotions_parsed"]
        if emo:
            for e in emotion_labels:
                emotion_avgs[e] += emo.get(e, 0)
            count += 1
    if count > 0:
        emotion_avgs = {e: round(v / count, 3) for e, v in emotion_avgs.items()}

    values = [emotion_avgs[e] for e in emotion_labels]
    values.append(values[0])
    fig.add_trace(
        go.Scatterpolar(r=values, theta=emotion_labels + [emotion_labels[0]],
                        fill="toself", line_color="#FFA15A"),
        row=2, col=2
    )

    fig.update_layout(
        title_text=f"Deep Dive: {topic_name.upper()}  ({len(topic_df)} posts)",
        height=700,
        showlegend=False,
        template="plotly_dark",
    )
    fig.show()

# Run deep dive for each topic
for topic in sorted(df["topic"].unique()):
    topic_deep_dive(topic)

In [13]:
# ══════════════════════════════════════════
# PANEL 10: Summary table
# ══════════════════════════════════════════

summary = []
for topic in sorted(df["topic"].unique()):
    t = df[df["topic"] == topic]
    dominant_emotion = "neutral"
    emo_sums = defaultdict(float)
    for _, row in t.iterrows():
        emo = row["emotions_parsed"]
        if emo:
            for e, v in emo.items():
                emo_sums[e] += v
    if emo_sums:
        dominant_emotion = max(emo_sums, key=emo_sums.get)

    summary.append({
        "Topic": topic,
        "Posts": len(t),
        "Avg Sentiment": round(t["sentiment_score"].mean(), 3),
        "% For": round((t["stance"] == "for").mean() * 100, 1),
        "% Against": round((t["stance"] == "against").mean() * 100, 1),
        "% Neutral": round((t["stance"] == "neutral").mean() * 100, 1),
        "Dominant Emotion": dominant_emotion,
        "Top Source": t["source_name"].value_counts().index[0],
    })

summary_df = pd.DataFrame(summary)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(summary_df.columns),
        fill_color="#1B6B93",
        font=dict(color="white", size=12),
        align="center",
    ),
    cells=dict(
        values=[summary_df[col] for col in summary_df.columns],
        fill_color="#2D2D2D",
        font=dict(color="white", size=11),
        align="center",
    ),
)])

fig.update_layout(
    title="Opinion Pulse — Summary Table",
    template="plotly_dark",
    height=400,
)
fig.show()

print("\n Dashboard complete! All panels rendered.")


 Dashboard complete! All panels rendered.
